<a href="https://colab.research.google.com/github/Shedjohn/Transformers_Building-blocks/blob/main/Notebook_1_%E2%80%94_Data_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install Packages**

In [1]:
!pip install datasets

In [2]:
import torch
import numpy as np
import random

from datasets import load_dataset
from collections import Counter
from torch.utils.data import Dataset, DataLoader

**Set Random Seeds**

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

**Load the Dataset**

In [5]:
dataset = load_dataset("nyu-mll/glue", "sst2")
dataset
dataset["train"][0]

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

{'sentence': 'hide new secretions from the parental units ',
 'label': 0,
 'idx': 0}

**Explore the Dataset**

In [6]:
print(len(dataset["train"]))
print(len(dataset["validation"]))
print(len(dataset["test"]))

for i in range(5):
    print(dataset["train"][i])

67349
872
1821
{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}
{'sentence': 'contains no wit , only labored gags ', 'label': 0, 'idx': 1}
{'sentence': 'that loves its characters and communicates something rather beautiful about human nature ', 'label': 1, 'idx': 2}
{'sentence': 'remains utterly satisfied to remain the same throughout ', 'label': 0, 'idx': 3}
{'sentence': 'on the worst revenge-of-the-nerds clichés the filmmakers could dredge up ', 'label': 0, 'idx': 4}


**Simple Tokenizer**

In [7]:
def tokenize(text):
    return text.lower().split()

**Build the Vocabulary**

In [8]:
counter = Counter()

for sample in dataset["train"]:
    counter.update(tokenize(sample["sentence"]))

vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word, freq in counter.items():
    vocab[word] = len(vocab)

print("Vocabulary size:", len(vocab))

Vocabulary size: 14818


**Create Reverse Vocabulary**

In [11]:
#Useful for debugging.
id_to_word = {idx: word for word, idx in vocab.items()}
print(id_to_word[5])

from


**Encode Sentences**

In [12]:
#Convert words into integers.
def encode(text):

    tokens = tokenize(text)

    ids = []

    for token in tokens:

        ids.append(
            vocab.get(token, vocab["<UNK>"])
        )

    return ids

**Padding**

In [14]:
#Transformers require equal-length sequences within a batch.
MAX_LEN = 40
def pad_sequence(ids):

    if len(ids) < MAX_LEN:

        ids = ids + [0] * (MAX_LEN - len(ids))

    else:

        ids = ids[:MAX_LEN]

    return ids

**Custom Dataset Class**

In [15]:
class SSTDataset(Dataset):

    def __init__(self, split):

        self.data = dataset[split]

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        sentence = self.data[idx]["sentence"]

        label = self.data[idx]["label"]

        ids = encode(sentence)

        ids = pad_sequence(ids)

        return (
            torch.tensor(ids),
            torch.tensor(label)
        )

**Create DataLoaders**

In [16]:
train_dataset = SSTDataset("train")
val_dataset = SSTDataset("validation")

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32
)

**Inspect One Batch**

In [17]:
batch = next(iter(train_loader))

inputs, labels = batch

print(inputs.shape)
print(labels.shape)

torch.Size([32, 40])
torch.Size([32])


**Verify the Pipeline**

In [18]:
#Recover a sentence from token IDs
ids = inputs[0].tolist()

words = [
    id_to_word.get(i, "<UNK>")
    for i in ids
]

print(words)

['more', 'revealing', ',', 'more', 'emotional', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>']
